# Hamiltonian Superposition Simulator

The simulator uses the general Hermitian two-state Hamiltonian

$$
\begin{aligned}
H&=h_0 I+\mathbf{h}\cdot\boldsymbol{\sigma},\\
H&=h_0 I+h_x\sigma_x+h_y\sigma_y+h_z\sigma_z
\end{aligned}
\qquad\qquad
H=
\begin{pmatrix}
h_0+h_z & h_x-i h_y\\
h_x+i h_y & h_0-h_z
\end{pmatrix}.
$$

The Cartesian parameters $(h_x,h_y,h_z)$ are displayed together with

$$
r=\sqrt{h_x^2+h_y^2+h_z^2},
\qquad
\theta=\operatorname{atan2}\!\left(\sqrt{h_x^2+h_y^2},h_z\right),
\qquad
\phi=\operatorname{atan2}(h_y,h_x).
$$

For $r>0$, the eigenvalues are $E_\pm=h_0\pm r$. The eigenstates and their
Born probabilities are reported in the fixed computational basis
$\{|0\rangle,|1\rangle\}$. A finite-shot laboratory-style acquisition is compared directly with
the exact Born probabilities.

**Dependencies:** NumPy, Matplotlib, ipywidgets, and IPython.

In [9]:

import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import ipywidgets as widgets

from IPython.display import display, HTML, Math, clear_output

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## Interactive simulator

Use the four visible sliders for $h_0$, $h_x$, $h_y$, and $h_z$. The numerical
readouts permit exact values as well as continuous slider adjustment. The default
$H=\sigma_x+\sigma_z$ gives nontrivial computational-basis probabilities so that
finite-$N$ sampling fluctuations are visible immediately.

The displayed analytic phase convention is

$$
|E_+\rangle=
\cos\left(\frac{\theta}{2}\right)|0\rangle+
e^{i\phi}\sin\left(\frac{\theta}{2}\right)|1\rangle,
$$

$$
|E_-\rangle=
-e^{-i\phi}\sin\left(\frac{\theta}{2}\right)|0\rangle+
\cos\left(\frac{\theta}{2}\right)|1\rangle.
$$

At $r=0$, the Hamiltonian is degenerate and does not select a unique
eigenbasis.

In [10]:

# ---------- Formatting helpers ----------

TOL = 1.0e-10

def clean_scalar(x, tol=TOL):
    x = float(np.real_if_close(x))
    return 0.0 if abs(x) < tol else x

def format_real(x, digits=4):
    x = clean_scalar(x)
    return f"{x:.{digits}f}"

def format_complex(z, digits=4, tol=TOL):
    z = complex(z)
    a = 0.0 if abs(z.real) < tol else z.real
    b = 0.0 if abs(z.imag) < tol else z.imag

    if abs(b) < tol:
        return f"{a:.{digits}f}"
    if abs(a) < tol:
        return f"{b:.{digits}f}i"
    sign = "+" if b >= 0 else "-"
    return f"{a:.{digits}f} {sign} {abs(b):.{digits}f}i"

def coeff_to_latex(z, digits=4, tol=TOL):
    z = complex(z)
    a = 0.0 if abs(z.real) < tol else z.real
    b = 0.0 if abs(z.imag) < tol else z.imag

    if abs(b) < tol:
        return f"{a:.{digits}f}"
    if abs(a) < tol:
        return f"{b:.{digits}f}i"
    sign = "+" if b >= 0 else "-"
    return f"\\left({a:.{digits}f}{sign}{abs(b):.{digits}f}i\\right)"

def state_to_latex(v, label):
    return (
        rf"{label}="
        rf"{coeff_to_latex(v[0])}\,|0\rangle"
        rf"{'+' if (v[1].real >= -TOL and abs(v[1].imag) < TOL) else '+'}"
        rf"{coeff_to_latex(v[1])}\,|1\rangle"
    ).replace("+-", "-")

def polar_component_latex(z, label):
    mag = abs(z)
    phase = np.angle(z)
    if mag < TOL:
        return rf"{label}: |c|=0,\ \arg(c)\ \text{{undefined}}"
    return rf"{label}: |c|={mag:.4f},\ \arg(c)={phase:.4f}\ \mathrm{{rad}}\ ({np.degrees(phase):.2f}^\circ)"

def hamiltonian(h0, hx, hy, hz):
    return np.array(
        [[h0 + hz, hx - 1j * hy],
         [hx + 1j * hy, h0 - hz]],
        dtype=complex,
    )

def spherical_parameters(hx, hy, hz):
    r = float(np.sqrt(hx**2 + hy**2 + hz**2))
    if r < TOL:
        return r, None, None
    theta = float(np.arctan2(np.sqrt(hx**2 + hy**2), hz))
    if np.sqrt(hx**2 + hy**2) < TOL:
        phi = None
    else:
        phi = float(np.arctan2(hy, hx))
    return r, theta, phi

def analytic_eigensystem(h0, hx, hy, hz):
    r, theta, phi = spherical_parameters(hx, hy, hz)
    if r < TOL:
        return {
            "degenerate": True,
            "r": r,
            "theta": None,
            "phi": None,
            "evals": np.array([h0, h0], dtype=float),
            "evec_minus": None,
            "evec_plus": None,
        }

    phi_for_state = 0.0 if phi is None else phi
    c = np.cos(theta / 2.0)
    s = np.sin(theta / 2.0)

    evec_plus = np.array([c, np.exp(1j * phi_for_state) * s], dtype=complex)
    evec_minus = np.array([-np.exp(-1j * phi_for_state) * s, c], dtype=complex)

    return {
        "degenerate": False,
        "r": r,
        "theta": theta,
        "phi": phi,
        "evals": np.array([h0 - r, h0 + r], dtype=float),
        "evec_minus": evec_minus,
        "evec_plus": evec_plus,
    }

def verify_eigensystem(H, result):
    if result["degenerate"]:
        return None
    checks = {}
    for name, E, v in [
        ("minus", result["evals"][0], result["evec_minus"]),
        ("plus", result["evals"][1], result["evec_plus"]),
    ]:
        checks[f"residual_{name}"] = np.linalg.norm(H @ v - E * v)
        checks[f"norm_{name}"] = abs(np.vdot(v, v) - 1.0)
    checks["orthogonality"] = abs(np.vdot(result["evec_minus"], result["evec_plus"]))
    return checks

# ---------- Controls ----------

# One sizing system only: percentage widths.
slider_layout = widgets.Layout(width="92%")
slider_style = {"description_width": "38px"}

def make_slider(description, value):
    return widgets.FloatSlider(
        value=value,
        min=-2.0,
        max=2.0,
        step=0.05,
        description=description,
        continuous_update=True,
        readout=True,
        readout_format=".3f",
        style=slider_style,
        layout=slider_layout,
    )

h0_slider = make_slider("h₀", 0.0)
hx_slider = make_slider("hₓ", 1.0)
hy_slider = make_slider("hᵧ", 0.0)
hz_slider = make_slider("h_z", 1.0)

shots_input = widgets.IntSlider(
    value=100,
    min=1,
    max=10000,
    step=1,
    description="Shots/experiment N",
    continuous_update=False,
    readout=True,
    style={"description_width": "125px"},
    layout=widgets.Layout(width="92%"),
)

seed_input = widgets.IntText(
    value=7,
    description="Seed",
    style={"description_width": "45px"},
    layout=widgets.Layout(width="150px"),
)

resample_button = widgets.Button(
    description="Resample",
    icon="refresh",
    button_style="",
)

reset_button = widgets.Button(
    description="Reset",
    icon="undo",
    button_style="",
)

sample_nonce = {"value": 0}

# Normal widget container for dynamically refreshed content.
dynamic_panel = widgets.VBox(
    layout=widgets.Layout(width="100%")
)


# ---------- Event handlers ----------

def reset_controls(_):
    h0_slider.value = 0.0
    hx_slider.value = 1.0
    hy_slider.value = 0.0
    hz_slider.value = 1.0
    shots_input.value = 100
    seed_input.value = 7

def resample(_):
    sample_nonce["value"] += 1
    update_display()

reset_button.on_click(reset_controls)
resample_button.on_click(resample)


# ---------- Main rendering ----------

def update_display(*_):
    h0 = float(h0_slider.value)
    hx = float(hx_slider.value)
    hy = float(hy_slider.value)
    hz = float(hz_slider.value)

    H = hamiltonian(h0, hx, hy, hz)
    result = analytic_eigensystem(h0, hx, hy, hz)

    r = result["r"]
    theta = result["theta"]
    phi = result["phi"]

    if result["degenerate"]:
        spherical_html = (
            f"<b>Spherical:</b> r = {r:.4f}; "
            "θ and φ are undefined because the Hamiltonian vector is zero."
        )
    else:
        phi_text = "undefined/irrelevant at the pole" if phi is None else (
            f"{phi:.4f} rad ({np.degrees(phi):.2f}°)"
        )
        spherical_html = (
            f"<b>Spherical:</b> r = {r:.4f}; "
            f"θ = {theta:.4f} rad ({np.degrees(theta):.2f}°); "
            f"φ = {phi_text}"
        )

    parameter_summary = widgets.HTML(
        value=(
            "<div style='padding:10px 12px;border:1px solid #bbb;border-radius:8px;"
            "margin:4px 0 10px 0;'>"
            f"<b>Cartesian:</b> (h₀,hₓ,hᵧ,h_z) = "
            f"({h0:.4f}, {hx:.4f}, {hy:.4f}, {hz:.4f})<br>"
            f"{spherical_html}"
            "</div>"
        ),
        layout=widgets.Layout(width="100%"),
    )

    matrix_latex = (
        r"H=\begin{pmatrix}"
        + format_complex(H[0, 0]) + " & " + format_complex(H[0, 1])
        + r"\\"
        + format_complex(H[1, 0]) + " & " + format_complex(H[1, 1])
        + r"\end{pmatrix}"
    )

    if not result["degenerate"]:
        phi_for_formula = 0.0 if phi is None else phi
        live_h_block = (
            rf"H={h0:.4f}I+({hx:.4f})\sigma_x+({hy:.4f})\sigma_y+({hz:.4f})\sigma_z"
            + rf"\;=\;{h0:.4f}I+{r:.4f}\Big["
            + rf"\sin({theta:.4f})\cos({phi_for_formula:.4f})\sigma_x+"
            + rf"\sin({theta:.4f})\sin({phi_for_formula:.4f})\sigma_y+"
            + rf"\cos({theta:.4f})\sigma_z\Big]"
        )
    else:
        live_h_block = (
            rf"H={h0:.4f}I+({hx:.4f})\sigma_x+({hy:.4f})\sigma_y+({hz:.4f})\sigma_z"
        )

    live_hamiltonian = widgets.HTMLMath(
        value=rf"""
        <div style="width:100%; margin:2px 0 10px 0; overflow-x:auto;">
          <div style="font-size:0.78em; text-align:center; white-space:nowrap;">
          \[
          {live_h_block}
          \]
          </div>
          <div style="font-size:0.92em; text-align:center; margin-top:2px;">
          \[
          {matrix_latex}
          \]
          </div>
        </div>
        """,
        layout=widgets.Layout(width="100%"),
    )

    if result["degenerate"]:
        E = result["evals"][0]

        degenerate_energy = widgets.HTMLMath(
            value=rf"""
            <div style="font-size:1.15em; margin:6px 0;">
            \(E_-=E_+={E:.4f}\)
            </div>
            """,
            layout=widgets.Layout(width="100%"),
        )

        degenerate_note = widgets.HTML(
            value=(
                "<div style='padding:12px;border-left:4px solid #777;margin:10px 0;'>"
                "<b>Degenerate Hamiltonian:</b> every orthonormal basis is an eigenbasis. "
                "The Hamiltonian does not select unique eigenstates, so eigenstate "
                "probabilities and sampling are not displayed."
                "</div>"
            ),
            layout=widgets.Layout(width="100%"),
        )

        dynamic_panel.children = (
            parameter_summary,
            live_hamiltonian,
            degenerate_energy,
            degenerate_note,
        )
        return

    Eminus, Eplus = result["evals"]
    vminus = result["evec_minus"]
    vplus = result["evec_plus"]

    p_plus = np.abs(vplus)**2
    p_minus = np.abs(vminus)**2

    N = int(shots_input.value)
    seed = int(seed_input.value) + 1000003 * sample_nonce["value"]
    rng = np.random.default_rng(seed)

    counts_minus = rng.multinomial(N, p_minus)
    counts_plus = rng.multinomial(N, p_plus)
    phat_minus = counts_minus / N
    phat_plus = counts_plus / N

    theory = np.array([p_minus[0], p_minus[1], p_plus[0], p_plus[1]])
    sample = np.array([phat_minus[0], phat_minus[1], phat_plus[0], phat_plus[1]])

    results_intro = widgets.HTML(
        value=(
            "<div style='font-size:1.0em;color:#555;margin:4px 0 8px 0;'>"
            "Energy eigenstates in the computational basis. Amplitudes and "
            "Born-rule probabilities use the same conditional labels as the histogram."
            "</div>"
        ),
        layout=widgets.Layout(width="100%"),
    )

    minus_col = widgets.HTMLMath(
        value=rf"""
        <div style="width:100%; box-sizing:border-box; font-size:0.98em;">
          <div style="font-weight:600;">Eigenvalue</div>
          <div>\(E_-={Eminus:.4f}\)</div>

          <div style="font-weight:600; margin-top:6px;">State</div>
          <div>\({state_to_latex(vminus, r'|E_-\rangle')}\)</div>

          <div style="font-weight:600; margin-top:6px;">Amplitude</div>
          <div>\(c_0={coeff_to_latex(vminus[0])}\)</div>
          <div>\(c_1={coeff_to_latex(vminus[1])}\)</div>

          <div style="font-weight:600; margin-top:6px;">Probability</div>
          <div>\(P(0|E_-)={p_minus[0]:.4f}\)</div>
          <div>\(P(1|E_-)={p_minus[1]:.4f}\)</div>

          <div style="margin-top:14px;">
          \(\Delta E=E_+-E_-=2r={2*r:.4f}\)
          </div>
        </div>
        """,
        layout=widgets.Layout(width="28%"),
    )

    plus_col = widgets.HTMLMath(
        value=rf"""
        <div style="width:100%; box-sizing:border-box; font-size:0.98em;">
          <div style="font-weight:600;">Eigenvalue</div>
          <div>\(E_+={Eplus:.4f}\)</div>

          <div style="font-weight:600; margin-top:6px;">State</div>
          <div>\({state_to_latex(vplus, r'|E_+\rangle')}\)</div>

          <div style="font-weight:600; margin-top:6px;">Amplitude</div>
          <div>\(c_0={coeff_to_latex(vplus[0])}\)</div>
          <div>\(c_1={coeff_to_latex(vplus[1])}\)</div>

          <div style="font-weight:600; margin-top:6px;">Probability</div>
          <div>\(P(0|E_+)={p_plus[0]:.4f}\)</div>
          <div>\(P(1|E_+)={p_plus[1]:.4f}\)</div>
        </div>
        """,
        layout=widgets.Layout(width="28%"),
    )

    labels = [
        r"$P(0|E_-)$",
        r"$P(1|E_-)$",
        r"$P(0|E_+)$",
        r"$P(1|E_+)$",
    ]

    x = np.arange(4)
    fig, ax = plt.subplots(figsize=(4.2, 4.2))

    ax.bar(
        x,
        sample,
        width=0.58,
        facecolor="none",
        edgecolor="black",
        linewidth=1.2,
        hatch="///",
    )

    for xi, p_theory in zip(x, theory):
        ax.hlines(
            p_theory,
            xi - 0.37,
            xi + 0.37,
            colors="black",
            linewidth=4.0,
            zorder=4,
        )

    ax.axvline(1.5, color="black", linewidth=0.8, linestyle=":", alpha=0.55)

    header_y_theory = -0.30
    header_y_sample = -0.235
    for xi, p_theory, p_sample in zip(x, theory, sample):
        ax.text(
            xi, header_y_theory, f"{p_theory:.3f}",
            transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8
        )
        ax.text(
            xi, header_y_sample, f"{p_sample:.3f}",
            transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8
        )

    ax.text(
        -0.48, header_y_theory, "theory",
        transform=ax.get_xaxis_transform(),
        ha="right", va="top", fontsize=8
    )
    ax.text(
        -0.48, header_y_sample, "sampled",
        transform=ax.get_xaxis_transform(),
        ha="right", va="top", fontsize=8
    )

    ax.set_xticks(x, labels)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Probability\ntheory & sampled")
    ax.set_title(
        f"Computational-basis sampling\nN={N} shots per eigenstate",
        pad=12,
    )
    ax.grid(axis="y", linewidth=0.5, alpha=0.25)
    fig.subplots_adjust(left=0.19, right=0.98, bottom=0.29, top=0.84)

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=110)
    plt.close(fig)

    plot_image = widgets.Image(
        value=buf.getvalue(),
        format="png",
        layout=widgets.Layout(width="100%"),
    )

    histogram_col = widgets.VBox(
        [plot_image],
        layout=widgets.Layout(width="42%"),
    )

    results_row = widgets.HBox(
        [minus_col, plus_col, histogram_col],
        layout=widgets.Layout(
            width="100%",
            justify_content="space-between",
            align_items="flex-start",
        ),
    )

    checks = verify_eigensystem(H, result)
    max_error = max(checks.values())

    verification = widgets.HTML(
        value=(
            "<div style='margin-top:8px;color:#555;font-size:0.9em;'>"
            f"Internal verification: maximum eigen-equation, normalization, or "
            f"orthogonality error = {max_error:.2e}"
            "</div>"
        ),
        layout=widgets.Layout(width="100%"),
    )

    dynamic_panel.children = (
        parameter_summary,
        live_hamiltonian,
        results_intro,
        results_row,
        verification,
    )


for control in [
    h0_slider, hx_slider, hy_slider, hz_slider,
    shots_input, seed_input
]:
    control.observe(update_display, names="value")


# ---------- Layout ----------

simulator_title = widgets.HTML(
    value="""
    <div style="text-align:center; margin:4px 0 12px 0;">
        <span style="font-size:22px; font-weight:600;">
        Two-Level Hamiltonian: Eigenstates, Superposition, and Measurement
        </span>
    </div>
    """,
    layout=widgets.Layout(width="100%"),
)

symbolic_hamiltonian = widgets.HTMLMath(
    value=r"""
    <div style="padding:10px 12px; margin:0 0 10px 0; border:1px solid #bbb; border-radius:8px;">
    <b>Hamiltonian controlled by the sliders</b><br><br>
    \[
    H=h_0 I+h_x\sigma_x+h_y\sigma_y+h_z\sigma_z
    =
    \begin{pmatrix}
    h_0+h_z & h_x-i h_y\\
    h_x+i h_y & h_0-h_z
    \end{pmatrix}.
    \]
    </div>
    """,
    layout=widgets.Layout(width="100%"),
)

controls_left = widgets.VBox(
    [
        widgets.HTML("<b>Hamiltonian controls</b>"),
        h0_slider,
        hx_slider,
        hy_slider,
        hz_slider,
    ],
    layout=widgets.Layout(width="45%"),
)

controls_right = widgets.VBox(
    [
        widgets.HTML("<b>Sampling controls — N shots for each eigenstate</b>"),
        shots_input,
        widgets.HBox(
            [seed_input, resample_button, reset_button],
            layout=widgets.Layout(width="100%"),
        ),
    ],
    layout=widgets.Layout(width="45%"),
)

controls_row = widgets.HBox(
    [controls_left, controls_right],
    layout=widgets.Layout(
        width="90%",
        margin="0 auto 8px auto",
        justify_content="space-between",
        align_items="flex-start",
    ),
)

dashboard = widgets.VBox(
    [
        simulator_title,
        symbolic_hamiltonian,
        controls_row,
        dynamic_panel,
    ],
    layout=widgets.Layout(width="100%"),
)

display(dashboard)
update_display()
